# Posterior-predictive video viewer

Experimental-versus-synthetic viewer for the posterior-predictive video check of
`srm-and-sbi-monomer-dimer-alp`. One engine
(`srm_and_sbi_monomer_dimer_alp/posterior_predictive_video_runner.py`) renders a synthetic video
beside an experimental recording and persists both in one self-contained clip
(`*_Synthetic_Video.npz`: the two videos plus the full provenance). Two thin entry points call it:
the biology one (`Script_Bank/Analysis/SRM_AND_SBI_MONOMER_DIMER_ALP_Posterior_Predictive_Video.py`)
and the detector one (`..._DETECTOR_Posterior_Predictive_Video.py`); the clips share one format and
this notebook views either. It only **views** a persisted clip: it never runs a simulation, so
scrubbing, zooming and re-running cells never regenerate anything. A new trajectory means
re-running the engine, a deliberate step, optionally with another `--seed`.

**What the synthetic panel is.** The clip names its own synthetic source (`synth_label`), printed
below and used as the panel title. In the visual check at a declared configuration (the biology
entry point with `--declared-rds`), the motion comes from the declared reaction-diffusion values,
one configuration for every recording with the receptor total at the prior center; the imaging is
the tagged `Nuisance_DLI` vector (`--nuisance-tag`, e.g. `REF`) plus the MET SCOPE camera values;
the labeling is the production labeling of the condition (the dye-count law at the declared
occupancy). Nothing is fitted to the recording: the comparison is a visual sanity check, read at
50 fps and through the pixel histograms of the static figure, not an estimate. With a MAP source
instead, the motion (biology) or the imaging (detector) comes from the recording's MAP estimate.

**How to run.** Pure viewer: it needs only `numpy`, `matplotlib` and `ipywidgets` (no project
package, no `MACHINE_PROFILE`, no ReaDDy). Render the clip with the entry point on the machine
that holds the data, copy the `*_Synthetic_Video.npz` here if needed, set `CLIP_PATH` in the
next code cell and launch `jupyter lab`.

**Color scaling.** Both panels always share ONE window, mirroring the engine's static figure, so
identical intensities map to identical colors and a given frame looks the same in the scrubber,
the player and the figure. By default (`autoscale`) the window is the shared min/max of the
displayed experimental-synthetic frame pair, recomputed per frame; `full` (the shared whole-clip
`[min, max]` of both clips, dim because one bright pixel anywhere sets the top) and `percentile`
(the shared whole-clip `[min, p99.99]`) are also available. Per-panel independent scaling is not
offered: identical colors would then no longer mean identical intensities. The data stay 16-bit.

The check's documentation, the list of all its files and the order of a check are in
`Script_Bank/Analysis/SRM_AND_SBI_MONOMER_DIMER_ALP_Posterior_Predictive_Video.md`.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, interact
# Pure viewer: needs only numpy + matplotlib + ipywidgets. No project package, no
# MACHINE_PROFILE, no ReaDDy -- rendering happens in the engine on the data machine.

In [ ]:
# Point CLIP_PATH at a *_Synthetic_Video.npz written by the posterior-predictive video engine
# (either entry point). The clip is self-contained (experimental + synthetic + provenance), so any
# absolute path works.
CLIP_PATH = "<absolute path to ..._Synthetic_Video.npz>"

# Display color scaling (data stay 16-bit; this only sets how the magma colormap is scaled).
# The experimental and synthetic panels ALWAYS share ONE window, in every mode, so identical
# intensities map to identical colors -- mirroring the engine's static figure (a given frame looks
# identical in the scrubber, the player and the figure). The mode only sets WHAT the shared window is:
#   "autoscale"  -> the shared min/max of the displayed frame pair, recomputed per frame (default).
#   "full"       -> the shared whole-clip [min, max] over BOTH clips; nothing clipped, but dim
#                   (one bright pixel anywhere in either clip sets the top).
#   "percentile" -> the shared whole-clip [min, p99.99] over both clips (cuts the top-0.01% hot-pixel
#                   sliver for contrast).
NORM_MODE = "autoscale"

import json
d = np.load(CLIP_PATH, allow_pickle=False)
experimental, synth = d["experimental"], d["synth"]        # (n_frames, H, W) uint16
n_frames = int(d["n_frames"])
fps = round(1.0 / float(d["frame_time_seconds"]))
SYNTH_LABEL = str(d["synth_label"])                        # the synthetic panel's title, as on the figure

def _rows(pairs, per_row=3):
    return "\n".join("  " + "  ".join(pairs[j:j + per_row]) for j in range(0, len(pairs), per_row))

# --- identity of this clip ---
print(f"{str(d['workflow'])} entry point  |  kind={str(d['kind'])} cell={int(d['cell'])}  "
      f"{n_frames} frames at {fps} fps  |  seed={int(d['seed'])}  |  engine version {str(d['package_version'])}")
print(f"experimental: {str(d['experimental_tif'])}")
print(f"synthetic:    {SYNTH_LABEL}")
print(f"experimental {experimental.shape}  synth {synth.shape}")
print(f"ADU  experimental  min {int(experimental.min())} median {int(np.median(experimental))} max {int(experimental.max())}")
print(f"ADU  synth         min {int(synth.min())} median {int(np.median(synth))} max {int(synth.max())}")

# --- provenance, as the static figure states it ---
# Imaging: the eleven values the render used (six emitter parameters + five SCOPE camera values).
_tag = str(d["nuisance_tag"])
print(f"imaging ({str(d['imaging_desc'])}):" if not _tag else f"imaging (tag {_tag}; {str(d['imaging_desc'])}):")
print(_rows([f"{k}={float(v):.4g}" for k, v in zip(d["imaging_keys"], d["imaging_physical"])]))
# Reaction-diffusion: the eleven values the motion came from -- declared (configuration file,
# name and SHA-256) or the recording's MAP estimate (biology), or the marginalized nuisance draw (detector).
_rds_src = str(d["rds_source"])
_rds = json.loads(str(d["rds_record_json"]))
_scn = _rds.get("scenario", {})
_head = f"reaction-diffusion ({_rds_src}"
if _scn.get("name"):
    _head += f": {_scn['name']}, {_rds['path']}, sha256 {_rds['sha256'][:12]}"
print(_head + "):")
_out = set(str(k) for k in d["rds_outside_prior"])
print(_rows([f"{k}={float(v):.4g}" + ("*" if str(k) in _out else "") for k, v in zip(d["rds_keys"], d["rds_provenance"])]))
if _out:
    print(f"  (* outside the biology prior: {sorted(_out)})")
# Labeling: the production labeling this render used, and the receptors it actually started with.
_lab = json.loads(str(d["labeling_record_json"]))
print(f"labeling: {str(d['condition'])} {_lab['law']['description']}, occupancy {_lab['occupancy']:.4g} "
      f"({_lab['occupancy_source']}); visible per subunit {_lab['visible_per_subunit']}")
_r = {str(k): int(round(float(v))) for k, v in zip(d["labeling_columns"], d["labeling_row"])}
_n_mono, _n_dim = _r["monomers_0"], _r["dimers_0"]
_v_mono, _v_dim = _r["monomers_visible_0"], _r["dimers_visible_0"]
print("initial receptors (frame 0, this render):\n"
      f"  {_r['n_subunits']} subunits = {_n_mono} monomers + {_n_dim} dimers ({_n_mono + _n_dim} receptors)\n"
      f"  labeled: {_r['n_labeled_subunits']} subunits carrying {_r['n_dyes']} dyes\n"
      f"  visible: {_v_mono} of {_n_mono} monomers + {_v_dim} of {_n_dim} dimers "
      f"({_r['dimers_two_labeled_0']} with both subunits labeled) = {_v_mono + _v_dim} spots")

# Shared windows over BOTH panels: the full range, and the whole-clip [min, p99.99].
_FULL = (float(min(experimental.min(), synth.min())), float(max(experimental.max(), synth.max())))
_PCTL = (float(min(experimental.min(), synth.min())),
         float(max(np.percentile(experimental, 99.99), np.percentile(synth, 99.99))))

def clim(exp2d, syn2d):
    """ONE shared (vmin, vmax) for BOTH panels -- identical intensities map to identical colors,
    mirroring the engine. 'full' the shared full-range [min, max] over both clips; 'percentile' the
    shared whole-clip [min, p99.99]; 'autoscale' the shared min/max of the displayed frame pair."""
    if NORM_MODE == "full":
        return _FULL
    if NORM_MODE == "percentile":
        return _PCTL
    return (float(min(exp2d.min(), syn2d.min())), float(max(exp2d.max(), syn2d.max())))

print(f"NORM_MODE={NORM_MODE}  " + (
    f"shared full-range window: {tuple(round(x) for x in _FULL)}" if NORM_MODE == "full"
    else (f"shared percentile window: {tuple(round(x) for x in _PCTL)}" if NORM_MODE == "percentile"
          else "(shared min/max of the displayed frame pair, recomputed per frame)")))


In [ ]:
# Scrub frames and zoom a shared region-of-interest into BOTH panels together.
# zoom = 1 shows the full frame; higher zoom crops tighter around (center x, center y).
H, W = experimental.shape[1], experimental.shape[2]

def _roi(frame, cx, cy, zoom):
    half_y, half_x = int(H / (2 * zoom)), int(W / (2 * zoom))
    y0, y1 = max(0, cy - half_y), min(H, cy + half_y)
    x0, x1 = max(0, cx - half_x), min(W, cx + half_x)
    return frame[y0:y1, x0:x1], (x0, x1, y0, y1)

def show(frame, cx, cy, zoom):
    ec, ext = _roi(experimental[frame], cx, cy, zoom)
    sc, _   = _roi(synth[frame], cx, cy, zoom)
    vmin, vmax = clim(ec, sc)                        # ONE shared window over both panels
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for a, crop, title in ((ax[0], ec, "EXPERIMENTAL"), (ax[1], sc, SYNTH_LABEL)):
        a.imshow(crop, cmap="magma", origin="lower", interpolation="none",
                 vmin=vmin, vmax=vmax, extent=ext)
        a.set_title(f"{title}   frame {frame}/{n_frames - 1}   zoom x{zoom:g}")
    plt.tight_layout(); plt.show()

interact(
    show,
    frame=IntSlider(min=0, max=n_frames - 1, step=1, value=n_frames // 2, description="frame"),
    cx=IntSlider(min=0, max=W - 1, step=1, value=W // 2, description="center x"),
    cy=IntSlider(min=0, max=H - 1, step=1, value=H // 2, description="center y"),
    zoom=FloatSlider(min=1.0, max=8.0, step=0.5, value=1.0, description="zoom"),
);

## Playback (real time)

Play both clips side by side at the native frame rate (50 fps for the production tier), using the
**same shared color scaling as the scrubber** (both panels on one window per frame pair, so a given
frame looks the same here, in the scrubber and in the static figure). Set `PLAY_ZOOM` (and the
center) to play a **cropped region** and check whether experimental and synthetic coincide locally.
The player embeds every frame losslessly (PNG frames), so a long clip builds a large widget: the
embed limit is raised below, and the cell reports the player's size and confirms that every frame
was embedded. Raise `PLAY_EVERY` only to lighten the build (playback stays real time).


In [ ]:
import matplotlib as mpl
from matplotlib import animation
from IPython.display import HTML

mpl.rcParams["animation.embed_limit"] = 512   # MB; a 1000-frame 2-panel player is ~390 MB (lossless PNG frames of
                                               # camera noise barely compress, ~290 KB per frame at any panel size,
                                               # plus a third for the base64 embedding)
PLAY_EVERY = 1                                 # stride: raise (2, 5, ...) to shrink the player; stays real-time

# Zoom for the player (same ROI convention as the scrubber): PLAY_ZOOM = 1 plays the full frame;
# raise it and set the center to play a cropped region and inspect local agreement.
PLAY_ZOOM = 1.0
PLAY_CX, PLAY_CY = W // 2, H // 2

def _pcrop(frame):
    return _roi(frame, PLAY_CX, PLAY_CY, PLAY_ZOOM)[0]

idx = list(range(0, n_frames, PLAY_EVERY))
fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
ec0, sc0 = _pcrop(experimental[0]), _pcrop(synth[0])
vmin, vmax = clim(ec0, sc0)                        # ONE shared window over both panels
panels = []
for a, c0, title in ((ax[0], ec0, "EXPERIMENTAL"), (ax[1], sc0, SYNTH_LABEL)):
    im = a.imshow(c0, cmap="magma", origin="lower", interpolation="none", vmin=vmin, vmax=vmax)
    a.set_title(title); a.set_xticks([]); a.set_yticks([]); panels.append(im)
plt.close(fig)   # suppress the static duplicate; the player below renders it

def _update(i):
    ec, sc = _pcrop(experimental[i]), _pcrop(synth[i])
    vmin, vmax = clim(ec, sc)                       # shared across both panels (fixed in full/percentile)
    for im, cf in zip(panels, (ec, sc)):
        im.set_data(cf); im.set_clim(vmin, vmax)
    fig.suptitle(f"frame {i}/{n_frames - 1}   ({fps} fps)   zoom x{PLAY_ZOOM:g}")
    return panels

anim = animation.FuncAnimation(fig, _update, frames=idx, interval=1000.0 * PLAY_EVERY / fps, blit=False)
player = anim.to_jshtml()
# Lossless PNG frames; every frame must be present -- matplotlib truncates silently above the embed limit.
_n_embedded = player.count("data:image/png;base64")
print(f"player: {len(player) / 1e6:.1f} MB, {_n_embedded} of {len(idx)} frames embedded"
      + ("" if _n_embedded == len(idx) else "  <-- TRUNCATED: raise animation.embed_limit or PLAY_EVERY"))
HTML(player)
